# TheSportsDB API Ingestion

Pull NFL team rosters, schedules, and event statistics from TheSportsDB free JSON API.

**Features:**
- Completely free JSON-based API
- Team rosters and player details
- Event (game) statistics
- League information and standings
- No authentication required for free endpoints

**Resources:**
- Website: https://www.thesportsdb.com
- API Docs: https://www.thesportsdb.com/api.php
- Free API: https://www.thesportsdb.com/api/v1/json/3/

**Note:** Free tier uses API key "3" for testing. For production use, consider Patreon supporter tier ($3/month) for higher rate limits.

In [0]:
import requests
import pandas as pd
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BASE_URL = "https://www.thesportsdb.com/api/v1/json/3"  # Free tier key
NFL_LEAGUE_ID = "4391"  # NFL league ID in TheSportsDB
SEASON = "2024"

print(f"📅 Fetching TheSportsDB data for Season {SEASON}")
print(f"API Endpoint: {BASE_URL}")
print(f"NFL League ID: {NFL_LEAGUE_ID}")
print("\nAvailable endpoints:")
print("  - /searchplayers.php?t={team} - Search players by team")
print("  - /eventspastleague.php?id={league_id} - Past events")
print("  - /lookupteam.php?id={team_id} - Team details")
print("  - /eventsseason.php?id={league_id}&s={season} - Events by season")

In [0]:
# First, get all NFL teams
print("Fetching NFL teams...")

try:
    url = f"{BASE_URL}/lookup_all_teams.php"
    params = {"id": NFL_LEAGUE_ID}
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    
    if 'teams' in data and data['teams']:
        teams = data['teams']
        teams_df = pd.DataFrame(teams)
        
        print(f"✓ Fetched {len(teams)} NFL teams")
        print(f"\nTeam names: {teams_df['strTeam'].tolist()[:10]}...")
        
        # Store team info for later use
        team_ids = {team['strTeam']: team['idTeam'] for team in teams}
        print(f"\nStored {len(team_ids)} team IDs")
    else:
        print("❌ No teams found")
        teams_df = pd.DataFrame()
        team_ids = {}
        
except Exception as e:
    print(f"❌ Error fetching teams: {e}")
    teams_df = pd.DataFrame()
    team_ids = {}

In [0]:
# Fetch all events (games) for the season
print("\nFetching events for season...")

try:
    url = f"{BASE_URL}/eventsseason.php"
    params = {
        "id": NFL_LEAGUE_ID,
        "s": SEASON
    }
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    
    if 'events' in data and data['events']:
        events = data['events']
        events_df = pd.DataFrame(events)
        
        print(f"✓ Fetched {len(events)} events")
        print(f"\nAvailable event columns: {list(events_df.columns)[:15]}...")
        print(f"\nSample event:")
        display(events_df[['strEvent', 'dateEvent', 'intHomeScore', 'intAwayScore']].head(5))
    else:
        print("❌ No events found")
        events_df = pd.DataFrame()
        
except Exception as e:
    print(f"❌ Error fetching events: {e}")
    events_df = pd.DataFrame()

In [0]:
# Fetch player rosters for teams
# Note: TheSportsDB has limited player statistics for NFL
# Focus on roster information

print("\nFetching player rosters...")

# Sample a few teams to get player data
sample_teams = ['Kansas City Chiefs', 'San Francisco 49ers', 'Buffalo Bills', 'Baltimore Ravens']

all_players = []

for team_name in sample_teams:
    if team_name in team_ids:
        try:
            print(f"  Fetching roster for {team_name}...")
            
            url = f"{BASE_URL}/searchplayers.php"
            params = {"t": team_name}
            
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            
            if 'player' in data and data['player']:
                players = data['player']
                print(f"    ✓ Found {len(players)} players")
                all_players.extend(players)
            else:
                print(f"    ⚠️ No players found")
                
        except Exception as e:
            print(f"    ❌ Error: {e}")

if all_players:
    players_df = pd.DataFrame(all_players)
    print(f"\n✓ Total players fetched: {len(players_df)}")
    print(f"\nPlayer columns: {list(players_df.columns)[:10]}...")
    print(f"\nSample players:")
    display(players_df[['strPlayer', 'strPosition', 'strTeam']].head(10))
else:
    print("\n⚠️ No player data available")
    players_df = pd.DataFrame()

In [0]:
# Transform player and event data to standard schema
# Note: TheSportsDB doesn't provide weekly fantasy points
# This is more useful for roster/team information

if 'events_df' in locals() and len(events_df) > 0:
    print("Transforming event data to Spark DataFrame...")
    
    rows = []
    for idx, event in events_df.iterrows():
        # Extract game information
        event_id = str(event.get('idEvent', ''))
        event_name = event.get('strEvent', '')
        date = event.get('dateEvent', '')
        home_team = event.get('strHomeTeam', '')
        away_team = event.get('strAwayTeam', '')
        home_score = int(event.get('intHomeScore', 0) or 0)
        away_score = int(event.get('intAwayScore', 0) or 0)
        
        # Convert to JSON for storage
        stats_json = json.dumps(event.to_dict(), default=str)
        
        rows.append(
            Row(
                event_id=event_id,
                event_name=event_name,
                date=date,
                home_team=home_team,
                away_team=away_team,
                home_score=home_score,
                away_score=away_score,
                season=int(SEASON),
                source="thesportsdb",
                stats=stats_json
            )
        )
    
    events_spark_df = spark.createDataFrame(rows)
    print(f"✓ Created Spark DataFrame with {events_spark_df.count()} events")
    display(events_spark_df.limit(10))
    
else:
    print("⚠️ No event data to transform")

In [0]:
# Write events to a separate bronze table for game data
if 'events_spark_df' in locals():
    print("Writing events to bronze_nfl_games...")
    
    bronze_events = events_spark_df.withColumn("ingested_at", F.current_timestamp())
    
    # Create or replace bronze_nfl_games table
    bronze_events.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("main.fantasai.bronze_nfl_games")
    
    print(f"✓ Appended {bronze_events.count()} events to bronze_nfl_games")
    
else:
    print("⚠️ No events to write")

In [0]:
%sql
-- Check TheSportsDB game data
SELECT 
  event_id,
  event_name,
  date,
  home_team,
  away_team,
  home_score,
  away_score,
  season,
  source
FROM main.fantasai.bronze_nfl_games
WHERE season = 2024 AND source = 'thesportsdb'
ORDER BY date DESC
LIMIT 20

## TheSportsDB API Features

### Available Endpoints (Free Tier)
1. **Leagues** - `/all_leagues.php` - All sports leagues
2. **Teams** - `/lookup_all_teams.php?id={league_id}` - Teams in league
3. **Players** - `/searchplayers.php?t={team}` - Player search by team
4. **Events** - `/eventsseason.php?id={league_id}&s={season}` - Events by season
5. **Team Details** - `/lookupteam.php?id={team_id}` - Detailed team info
6. **Event Details** - `/lookupevent.php?id={event_id}` - Detailed event info

### Pricing
- **Free Tier**: API key "3" - Limited requests, public testing
- **Patreon ($3/month)**: Higher rate limits, private API key
- **Patreon ($10/month)**: Highest rate limits, priority support

### Key Advantages
- ✅ **Completely free** for basic usage
- ✅ **No authentication** required for testing
- ✅ Team rosters and player information
- ✅ Game schedules and results
- ✅ Multiple sports coverage

### Limitations
- ❌ **No weekly player statistics** for fantasy football
- ❌ Limited to team/game level data
- ❌ Not real-time during games
- ❌ Rate limits on free tier

### Best Use Cases
- Team roster management
- Game schedules and results
- Historical team/player information
- Building team reference data

### Recommendation for Fantasy Football
**TheSportsDB is NOT ideal for fantasy football player stats.**

Better options:
- **nflverse** - Free, comprehensive weekly player stats
- **Fantasy Football Data Pros** - Free, fantasy-focused API

**Use TheSportsDB for:**
- Team logos and branding
- Game schedules
- Roster information
- Supplemental team data